# Digital Usage Clusters

### Imports

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist, pdist

### Data loading and preprocessing

In [ ]:
# Load data
df = pd.read_csv("../data/stu_filtered_270425.csv")

# Select only relevant numeric features
numeric_features = [
        'active_months',
        'total_months',
        'sessions_per_month',
        'sessions_in_peak_month',
        'gini_monthly'
]

df_numeric = df[numeric_features]

# Impute missing values with median
imputer = SimpleImputer(strategy='median')
df_imputed = pd.DataFrame(imputer.fit_transform(df_numeric), columns=df_numeric.columns)

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_imputed)

### K-means modelling

We will use several indices to decide which is the best-fitting model, including the inertias, silhouettes, Calinski-Harabasz Davies-Bouldin and Dunn scores. Given that it was consensually decided that an acceptable and interpretable solution could not contain more than nine different clusters, I compared all solutions from K ranging from 2 to 9.

In [ ]:
def dunn_index(X, labels):
    """Compute the Dunn Index for cluster labels"""
    clusters = np.unique(labels)
    # Inter-cluster distances (between centroids)
    centroids = np.array([X[labels == c].mean(axis=0) for c in clusters])
    inter_dists = cdist(centroids, centroids)
    np.fill_diagonal(inter_dists, np.inf)  # ignore self-distance
    min_intercluster = np.min(inter_dists)

    # Intra-cluster distances (cluster diameters)
    diameters = []
    for c in clusters:
        cluster_points = X[labels == c]
        if len(cluster_points) > 1:
            diam = np.max(pdist(cluster_points))
        else:
            diam = 0
        diameters.append(diam)
    max_diameter = np.max(diameters)

    return min_intercluster / max_diameter if max_diameter > 0 else np.nan

inertias = []
silhouettes = []
ch_scores = []
db_scores = []
dunn_scores = []
K = range(2, 9)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(X_scaled)

    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))
    ch_scores.append(calinski_harabasz_score(X_scaled, labels))
    db_scores.append(davies_bouldin_score(X_scaled, labels))
    dunn_scores.append(dunn_index(X_scaled, labels))

# Make a table
results = pd.DataFrame({
    'k': K,
    'Inertia': inertias,
    'Silhouette': silhouettes,
    'Calinski-Harabasz': ch_scores,
    'Davies-Bouldin': db_scores,
    'Dunn Index': dunn_scores
})

print(results)

# Plot elbow and silhouette
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(K, inertias, 'bo-')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia (SSE)')
plt.title('Elbow Method')

plt.subplot(1, 2, 2)
plt.plot(K, silhouettes, 'ro-')
plt.xlabel('Number of clusters')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs. k')

plt.tight_layout()
plt.show()

After deciding that the best fitting model was the one comprising six clusters, we provide the metrics for the model and summarize the data for each of the clusters.

In [ ]:
best_k = 6
kmeans = KMeans(n_clusters=best_k, random_state=42)
df_imputed["cluster"] = kmeans.fit_predict(X_scaled)

In [ ]:
# Compute inertia
inertia = kmeans.inertia_

# Compute silhouette score
silhouette = silhouette_score(X_scaled, df_imputed["cluster"])

print(f"Inertia: {inertia}")
print(f"Silhouette Score: {silhouette}")

In [ ]:
# Compute mean of each feature per cluster
cluster_summary = df_imputed.groupby("cluster").mean()
print(cluster_summary)

# Optional: Visualize with PCA
from sklearn.decomposition import PCA
import seaborn as sns

pca = PCA(n_components=2)
components = pca.fit_transform(X_scaled)

df_plot = pd.DataFrame(components, columns=["PC1", "PC2"])
df_plot["cluster"] = df_imputed["cluster"]

plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_plot, x="PC1", y="PC2", hue="cluster", palette="tab10")
plt.title("K-Means Clusters (PCA-reduced)")
plt.show()

In [ ]:
cluster_counts = df_imputed["cluster"].value_counts().sort_index()
print(cluster_counts)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Define the cluster name mapping
cluster_names = {
    0: "One-off users",
    1: "Intensive short-term users",
    2: "Consistent Moderate users",
    3: "Long-term users",
    4: "Minimal users",
    5: "Bursty moderate users"
}

# Count students per cluster
cluster_counts = df_imputed["cluster"].value_counts().sort_index()

# Define the custom order
custom_order = [0, 4, 1, 5, 2, 3]

# Create a DataFrame for plotting in the desired order
plot_df = pd.DataFrame({
    "Cluster": [cluster_names[i] for i in custom_order],
    "Count": [cluster_counts[i] for i in custom_order]
})

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(data=plot_df, x="Cluster", y="Count", palette="tab10", order=plot_df["Cluster"])
plt.title("Number of Students per Usage Cluster")
plt.xticks(rotation=30, ha='right')
plt.ylabel("Number of Students")
plt.xlabel("Usage Cluster")
plt.tight_layout()
plt.show()